# Ekstraksi Fitur Warna (RGB Mean, Std, Skewness) dan Tekstur (GLCM) serta Label

Notebook ini mengekstrak fitur visual lengkap dari setiap citra:
1. **Fitur Warna (Color Moments orde 1, 2, dan 3)**:
   - `R_mean`, `G_mean`, `B_mean` (Rata-rata intensitas warna per kanal)
   - `R_std`, `G_std`, `B_std` (Standar deviasi variasi warna per kanal)
   - `R_skew`, `G_skew`, `B_skew` (Skewness kemencengan distribusi warna per kanal)
   - `gray_skew` (Skewness pada citra keabuan/grayscale)
2. **Fitur Tekstur GLCM (Gray-Level Co-occurrence Matrix Haralick)**:
   - `glcm_contrast`: Kontras variasi intensitas lokal antar pasangan piksel
   - `glcm_dissimilarity`: Perbedaan nilai tingkat keabuan
   - `glcm_homogeneity`: Keseragaman lokal distribusi matriks
   - `glcm_energy`: Keseragaman tekstur (akar dari ASM)
   - `glcm_correlation`: Korelasi ketergantungan spasial nilai piksel
   - `glcm_asm`: Angular Second Moment

Label diambil dari folder kelas `RAPI` dan `BERANTAKAN`. Hasilnya disimpan sebagai `fitur_citra_lengkap.csv` dan `fitur_rgb_std.csv`.

## 1. Import library dan tentukan folder data

Jika notebook dan folder `RAPI` serta `BERANTAKAN` berada di folder yang sama, konfigurasi otomatis menggunakan folder kerja. Jika tidak, ubah `BASE_DIR` ke lokasi folder dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

BASE_DIR = Path.cwd()
NAMA_FOLDER_KELAS = ("RAPI", "BERANTAKAN")

if not all((BASE_DIR / nama).is_dir() for nama in NAMA_FOLDER_KELAS):
    BASE_DIR = Path(r"C:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra")

if not all((BASE_DIR / nama).is_dir() for nama in NAMA_FOLDER_KELAS):
    raise FileNotFoundError(
        "Folder RAPI dan BERANTAKAN tidak ditemukan. Ubah BASE_DIR ke folder dataset."
    )

print(f"Folder dataset: {BASE_DIR}")

Folder dataset: c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra


## 2. Ekstraksi Fitur Warna dan Tekstur GLCM

Citra diubah ke **RGB** untuk menghitung Mean, Standar Deviasi, dan Skewness tiap kanal. Citra juga diubah ke **Grayscale** untuk menghitung matriks GLCM 4 arah (0 derajat, 45 derajat, 90 derajat, 135 derajat) dengan jarak d=1 dan mengekstrak fitur statistik Haralick.

In [2]:
def extract_glcm_features(gray):
    """
    Menghitung GLCM 4 arah (0, 45, 90, 135 derajat) jarak d=1
    dan mengekstrak 6 fitur tekstur Haralick utama.
    """
    levels = 256
    glcm = np.zeros(levels * levels, dtype=np.int32)

    # 0 derajat (horizontal)
    glcm += np.bincount(((gray[:, :-1].astype(np.int32) << 8) | gray[:, 1:]).ravel(), minlength=65536)
    # 90 derajat (vertikal)
    glcm += np.bincount(((gray[:-1, :].astype(np.int32) << 8) | gray[1:, :]).ravel(), minlength=65536)
    # 45 derajat (diagonal kanan)
    glcm += np.bincount(((gray[:-1, :-1].astype(np.int32) << 8) | gray[1:, 1:]).ravel(), minlength=65536)
    # 135 derajat (diagonal kiri)
    glcm += np.bincount(((gray[:-1, 1:].astype(np.int32) << 8) | gray[1:, :-1]).ravel(), minlength=65536)

    # Jadikan simetris & normalisasi
    glcm = glcm.reshape((levels, levels)).astype(np.float64)
    glcm += glcm.T
    s = glcm.sum()
    if s > 0:
        glcm /= s

    # Ekstraksi fitur statistik Haralick
    i, j = np.indices((levels, levels), dtype=np.float64)
    contrast = float(np.sum(glcm * ((i - j) ** 2)))
    dissimilarity = float(np.sum(glcm * np.abs(i - j)))
    homogeneity = float(np.sum(glcm / (1.0 + (i - j) ** 2)))
    asm = float(np.sum(glcm ** 2))
    energy = float(np.sqrt(asm))

    mu_i = np.sum(i * glcm)
    mu_j = np.sum(j * glcm)
    var_i = np.sum(glcm * ((i - mu_i) ** 2))
    var_j = np.sum(glcm * ((j - mu_j) ** 2))
    std_prod = np.sqrt(var_i * var_j)
    correlation = float(np.sum(glcm * (i - mu_i) * (j - mu_j)) / std_prod if std_prod > 0 else 1.0)

    return {
        "glcm_contrast": contrast,
        "glcm_dissimilarity": dissimilarity,
        "glcm_homogeneity": homogeneity,
        "glcm_energy": energy,
        "glcm_correlation": correlation,
        "glcm_asm": asm,
    }


LABEL_MAP = {
    "RAPI": "rapi",
    "BERANTAKAN": "berantakan",
}
EKSTENSI_GAMBAR = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

NAMA_FITUR_WARNA = [
    "R_mean", "G_mean", "B_mean",
    "R_std", "G_std", "B_std",
    "R_skew", "G_skew", "B_skew",
    "gray_skew"
]
NAMA_FITUR_GLCM = [
    "glcm_contrast", "glcm_dissimilarity", "glcm_homogeneity",
    "glcm_energy", "glcm_correlation", "glcm_asm"
]
NAMA_FITUR = [*NAMA_FITUR_WARNA, *NAMA_FITUR_GLCM]

baris_hasil = []

for nama_folder, label in LABEL_MAP.items():
    folder_kelas = BASE_DIR / nama_folder
    if not folder_kelas.is_dir():
        raise FileNotFoundError(f"Folder kelas tidak ditemukan: {folder_kelas}")

    daftar_gambar = sorted(
        path for path in folder_kelas.rglob("*")
        if path.is_file() and path.suffix.lower() in EKSTENSI_GAMBAR
    )

    for path_gambar in daftar_gambar:
        with Image.open(path_gambar) as gambar:
            rgb = np.asarray(gambar.convert("RGB"), dtype=np.float32)
            gray = np.asarray(gambar.convert("L"), dtype=np.uint8)

        # 1. Fitur Warna: Mean dan Std
        mean_rgb = rgb.mean(axis=(0, 1))
        std_rgb = rgb.std(axis=(0, 1), ddof=0)

        # 2. Fitur Skewness: Warna (RGB) dan Keabuan (Grayscale)
        diff_rgb = rgb - mean_rgb
        std_safe = np.where(std_rgb == 0, 1e-10, std_rgb)
        skew_rgb = np.mean(diff_rgb ** 3, axis=(0, 1)) / (std_safe ** 3)

        diff_gray = gray.astype(np.float32) - gray.mean()
        std_gray = gray.std()
        skew_gray = float(np.mean(diff_gray ** 3) / (std_gray ** 3) if std_gray > 0 else 0.0)

        # 3. Fitur Tekstur GLCM
        glcm_feats = extract_glcm_features(gray)

        baris_hasil.append({
            "file_name": path_gambar.name,
            "label": label,
            "R_mean": mean_rgb[0],
            "G_mean": mean_rgb[1],
            "B_mean": mean_rgb[2],
            "R_std": std_rgb[0],
            "G_std": std_rgb[1],
            "B_std": std_rgb[2],
            "R_skew": skew_rgb[0],
            "G_skew": skew_rgb[1],
            "B_skew": skew_rgb[2],
            "gray_skew": skew_gray,
            **glcm_feats,
        })

if not baris_hasil:
    raise ValueError(f"Tidak ada gambar yang didukung di {BASE_DIR}")

df_fitur = pd.DataFrame(baris_hasil, columns=["file_name", "label", *NAMA_FITUR])
print(f"Total gambar berhasil diekstrak: {len(df_fitur)}")
print(f"Total kolom fitur: {len(NAMA_FITUR)}")
display(df_fitur.head())

Total gambar berhasil diekstrak: 128
Total kolom fitur: 16


,file_name,label,R_mean,G_mean,B_mean,R_std,G_std,B_std,R_skew,G_skew,B_skew,gray_skew,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation,glcm_asm
0,WhatsApp Image 2026-11-08 at 17.44.49.jpeg,rapi,145.480087,140.709961,134.473236,56.314926,57.835003,56.640064,-0.553174,-0.633935,-0.734334,-0.602398,309.679075,6.719302,0.457421,0.034463,0.951506,0.001188
1,WhatsApp Image 2026-11-08 at 17.44.50.jpeg,rapi,150.974564,143.231720,130.010712,49.928535,54.269417,57.145000,-0.568053,-0.425871,-0.555725,-0.421551,297.213213,6.879885,0.386348,0.031195,0.944639,0.000973
2,WhatsApp Image 2026-11-08 at 17.44.52.jpeg,rapi,144.009720,136.665451,129.093994,53.460590,60.005260,66.150597,-0.599536,-0.542911,-0.534105,-0.530127,294.951413,6.603947,0.396080,0.028414,0.955348,0.000807
3,WhatsApp Image 2026-11-08 at 17.44.54.jpeg,rapi,143.930969,136.801254,124.905731,53.039989,57.443146,63.650955,-0.567538,-0.334346,-0.437164,-0.336468,348.482349,7.723554,0.310959,0.022467,0.941423,0.000505
4,WhatsApp Image 2026-11-08 at 17.44.57.jpeg,rapi,143.155655,140.716522,134.892151,53.967888,59.390045,62.718475,-0.577713,-0.618788,-0.543752,-0.590694,282.787003,6.484796,0.404144,0.028978,0.956750,0.000840


## 3. Periksa label dan simpan hasil

Kolom fitur kini berjumlah 16 (10 fitur warna/intensitas + 6 fitur tekstur GLCM). Hasil disimpan ke file CSV.

In [3]:
print("Jumlah gambar per label:")
display(df_fitur["label"].value_counts().rename_axis("label").to_frame("jumlah"))

kolom_csv = ["file_name", "label", *NAMA_FITUR]

# Simpan ke fitur_citra_lengkap.csv dan perbarui fitur_rgb_std.csv
file_output_lengkap = BASE_DIR / "fitur_citra_lengkap.csv"
df_fitur[kolom_csv].to_csv(file_output_lengkap, index=False)

file_output_lama = BASE_DIR / "fitur_rgb_std.csv"
df_fitur[kolom_csv].to_csv(file_output_lama, index=False)

print(f"\nCSV berhasil disimpan:")
print(f"- {file_output_lengkap}")
print(f"- {file_output_lama}")
print(f"Jumlah total kolom: {len(kolom_csv)} (2 metadata + {len(NAMA_FITUR)} fitur)")

Jumlah gambar per label:


,jumlah
label,
berantakan,71
rapi,57



CSV berhasil disimpan:
- c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra\fitur_citra_lengkap.csv
- c:\Users\ASUS\OneDrive\画像\ドキュメント\1.PERKULIAHAN\SEMESTER 5\1.Proyek Sain Data\Citra\fitur_rgb_std.csv
Jumlah total kolom: 18 (2 metadata + 16 fitur)
